# Spam Classifier
## Assignment Preamble
Please ensure you carefully read all of the details and instructions on the assignment page, this section, and the rest of the notebook. If anything is unclear at any time please post on the forum or ask a tutor well in advance of the assignment deadline.

In addition to all of the instructions in the body of the assignment below, you must also follow the following technical instructions for all assignments in this unit. *Failure to do so may result in a grade of zero.*
* [At the bottom of the page](#Submission-Test) is some code which checks you meet the submission requirements. You **must** ensure that this runs correctly before submission.
* Do not modify or delete any of the cells that are marked as test cells, even if they appear to be empty.
* Do not duplicate any cells in the notebook – this can break the marking script. Instead, insert a new cell (e.g. from the menu) and copy across any contents as necessary.

Remember to save and backup your work regularly, and double-check you are submitting the correct version.

This notebook is the primary reference for your submission. You may write code in separate `.py` files but it must be clearly imported into the notebook so that it runs without needing to reference those files, and you must explain clearly what functionality is contained in those files (through comments, markdown cells, etc).

As always, **the work you submit for this assignment must be entirely your own.** Do not copy or work with other students. Do not copy answers that you find online. These assignments are designed to help improve your understanding first and foremost – the process of doing the assignment is part of *learning*. They are also used to assess your ability, and so you must uphold academic integrity. Submitting plagiarised work risks your entire place on your degree.

**The pass mark for this assignment is 40%.** We expect that students, on average, will be able to produce a submission which gets a mark between 50-70% within the normal workload allocation for the unit, but this will vary depending on individual backgrounds. Please ask for help if you are struggling.

## Getting Started
Spam refers to unwanted email, often in the form of advertisements. In the literature, an email that is **not** spam is called *ham*. Most email providers offer automatic spam filtering, where spam emails will be moved to a separate inbox based on their contents. Of course this requires being able to scan an email and determine whether it is spam or ham, a classification problem. This is the subject of this assignment.

This assignment has two parts. Each part is worth 50% of the overall grade for this assignment.

For part one you will write a supervised learning based classifier to determine whether a given email is spam or ham. You must write and submit the code in this notebook. The training data is provided for you. You may use any classification method. Marks will be awarded primarily based on the accuracy of your classifier on unseen test data, but there are also marks for estimating how accurate you think your classifier will be.

In part two you will produce a short video explaining your implementation, any decisions or extensions you made, and what parameter values you used. This part is explained in more detail on the assignment page. The video file must be submitted with your assignment.

### Choice of Algorithm
While the classification method is a completely free choice, the assignment folder includes [a separate notebook file](data/naivebayes.ipynb) which can help you implement a Naïve Bayes solution. If you do use this notebook, you are still responsible for porting your code into *this* notebook for submission. A good implementation should give a high  enough accuracy to get a good grade on this section (50-70%).

You could also consider a k-nearest neighbour algorithm, but this may be less accurate. Logistic regression is another option that you may wish to consider.

If you are looking to go beyond the scope of the unit, you might be interested in building something more advanced, like an artificial neural network. This is possible just using `numpy`, but will require significant self-directed learning. *Extensions like this are left unguided and are not factored into the unit workload estimates.*

**Note:** you may use helper functions in libraries like `numpy` or `scipy`, but you **must not** import code which builds entire models for you. This includes but is not limited to use of libraries like `scikit-learn`, `tensorflow`, or `pytorch` – there will be plenty of opportunities for these libraries in later units. The point of this assignment is to understand code the actual algorithm yourself. ***If you are in any doubt about any particular library or function please ask a tutor.*** Submissions which ignore this will receive penalties or even zero marks.

If you choose to implement more than one algorithm, please feel free to include your code and talk about it in part two (your video presentation), but only the code in this notebook will be used in the automated testing.

## Training Data
The training data is described below and has 1000 rows. There is also a 500 row set of test data. These are functionally identical to the training data, they are just in a separate csv file to encourage you to split out your training and test data. You should consider how to best make use of all available data without overfitting, and to help produce an unbiased estimate for your classifier's accuracy.

The cell below loads the training data into a variable called `training_spam`.

In [1]:
import numpy as np
from IPython.display import HTML,Javascript, display

training_spam = np.loadtxt(open("data/training_spam.csv"), delimiter=",").astype(int)
print("Shape of the spam training data set:", training_spam.shape)
print(training_spam)

Shape of the spam training data set: (1000, 55)
[[1 0 0 ... 0 0 0]
 [0 0 1 ... 1 0 0]
 [0 0 0 ... 1 0 0]
 ...
 [0 0 0 ... 0 0 1]
 [1 1 1 ... 1 1 0]
 [1 0 0 ... 1 1 1]]


Your training set consists of 1000 rows and 55 columns. Each row corresponds to one email message. The first column is the _response_ variable and describes whether a message is spam `1` or ham `0`. The remaining 54 columns are _features_ that you will use to build a classifier. These features correspond to 54 different keywords (such as "money", "free", and "receive") and special characters (such as ":", "!", and "$"). A feature has the value `1` if the keyword appears in the message and `0` otherwise.

As mentioned there is also a 500 row set of *test data*. It contains the same 55 columns.

In [2]:
testing_spam = np.loadtxt(open("data/testing_spam.csv"), delimiter=",").astype(int)
print("Shape of the spam testing data set:", testing_spam.shape)
print(testing_spam)

Shape of the spam testing data set: (500, 55)
[[1 0 0 ... 1 1 1]
 [1 1 0 ... 1 1 1]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 1 0 0]]


## Part One
Write all of the code for your classifier below this cell. There is some very rough skeleton code in the cell directly below. You may insert more cells below this if you wish, but you must not duplicate any cells as this can break the grading script.

### Submission Requirements
Your code must provide a variable with the name `classifier`. This object must have a method called `predict` which takes input data and returns class predictions. The input will be a single $n \times 54$ numpy array, your classifier should return a numpy array of length $n$ with classifications. There is a demo in the cell below, and a test you can run before submitting to check your code is working correctly.

Your code must run on our test machine in under 30 seconds. If you wish to train a more complicated model (e.g. neural network) which will take longer, you are welcome to save the model's weights as a file and then load these in the cell below so we can test it. You must include the code which computes the original weights, but this must not run when we run the notebook – comment out the code which actually executes the routine and make sure it is clear what we need to change to get it to run. Remember that we will be testing your final classifier on additional hidden data.

In [7]:
# ---Stage 1: Inspect data and k-fold cross-validation set up---

from typing import Callable
import numpy as np

def split_x_and_y(data: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """
    Splits input data table (with n rows) into features (X) and response (y).
   
    Args:
        data: a data table, shape (n, 55). Column 0 contains the labels (0 = ham, 1 = spam) and
            Columns 1-54 hold the features corresponding to 54 different keywords. 
   
    Returns:
        X: features, shape (n,54)
        y: labels, shape (n, )
    """

    X = data[:, 1:] # all rows and columns 1 to the last one 
    y = data[:, 0] #all rows and only column 0
    return X, y

def inspect_data(X: np.ndarray, y:np.ndarray, top: int = 10) -> tuple[np.ndarray, np.ndarray]:
    """
    Prints a summary of the data including the spam vs ham balance, baseline accuracy, 
    keywords with only one label type and the keywords whose frequency differs most between spam vs ham.

    Args:
        X: features, shape (n, 54)
        y: labels, shape (n, )
        top: the number of keywords to include in each list (top ham keywords and top spam keywords)
   
    Returns:
        p_ham:  P(keyword present| ham) for each keyword, shape(54, )
        p_spam: P(keyword present| spam) for each keyword, shape(54, )
    """
    n_emails, n_features = X.shape

    n_spam = int(y.sum())
    n_ham = n_emails - n_spam
    spam_rate = n_spam/n_emails
    ham_rate = n_ham/n_emails

    print(f"{n_emails} emails, {n_features} features")
    print(f"spam: {n_spam} ({spam_rate:.1%}) ham: {n_ham} ({ham_rate:.1%})")

    # If a model always predicts the more common class thant that is like having no spam filtering.
    # Androutsopoulos et al (2000) advise to always compare model performance against this. 
    baseline_accuracy = max(spam_rate, ham_rate)
    print(f"a model that always predicts the more common class has baseline accuracy of: {baseline_accuracy:.3f}")

    # Share of ham and spam emails that contain each keyword. 
    p_ham  =  X[y == 0].mean(axis=0)   #P(keyword present| ham)
    p_spam =  X[y == 1].mean(axis=0)   #P(keyword present| spam)

    # If >0: keyword is more common in spam, if <0: keyword is more common in ham
    spam_diff_ham = p_spam - p_ham  
    # The keyword column numbers, sorted from most associated with ham (smallest values) to most associated with spam (largest values)
    order = np.argsort(spam_diff_ham) 

    header = f"{'keyword':>8} {'P(x|ham)':>10} {'P(x|spam)':>10} {'spam_diff_ham':>14}"

    print("strongest spam indicating keywords")
    print(header)
    # need to reverse the order so spam comes first
    for j in order[::-1][:top]:
        print(f"{j:>9} {p_ham[j]:>10.3f} {p_spam[j]:>10.3f} {spam_diff_ham[j]:>+14.2f}")

    print("\nstrongest ham indicating keywords")
    print(header)
    for j in order[:top]:
        print(f"{j:>9} {p_ham[j]:>10.3f} {p_spam[j]:>10.3f} {spam_diff_ham[j]:>+14.2f}")

    # Are there any keywords that are never in the spam or never in the ham emails?
    # These would result in a probability of 0 and log(0) is undefined.
    # This is a problem for Naive Bayes but we can correct this later with 
    # Laplace smoothing (Russell and Norvig, 2022, §24.1.4). 

    never_in_ham = np.flatnonzero(p_ham == 0)
    never_in_spam = np.flatnonzero(p_spam == 0)

    print(f"\nkeywords that never appear in the ham emails: {never_in_ham}")
    print(f"\nkeywords that never appear in the spam emails: {never_in_spam}")

    return p_ham, p_spam

# Stratified k-fold cross-validation (Kohavi, 1995; Russell and Norvig, 2022, §19.4). 
# The emails are divided into k folds that are stratified by label.
# I use k = 10 here as this was found by Kohavi (1995) to be the best-performing setting
# for model selection working with real-world datasets. 
# Cross-validation means each fold gets used once to test 
# the model (trained on the other k-1 folds) so every email will be used in testing once. 

def strat_k_fold(y: np.ndarray, k: int, rng: np.random.Generator) -> list[np.ndarray]:
    """
    Divides the emails into k folds that each have approximately the same ham to spam ratio.

    Args:
        y: labels, shape (n, )
        k: number of folds
        rng: seeded random number generator for reproducibility 
    
    Returns:
        list of k arrays where each array contains the row numbers of the emails in that fold. 
    """
    folds = [[] for _ in range(k)]  # k empty folds
    for label in np.unique(y):
        # row numbers of each email with this label
        label_rows = np.flatnonzero(y == label) 

        rng.shuffle(label_rows)

        # divide into k approximately equal shares
        # each ith share goes into the ith fold
        for i, share in enumerate(np.array_split(label_rows, k)):
            folds[i].extend(share)
    return [np.array(fold) for fold in folds]

def cross_validate(
        model_to_test: Callable[[], object],
        X: np.ndarray,
        y: np.ndarray,
        seed: int = 0,
        k: int = 10,
        verbose: bool = True
) -> tuple[float, float, np.ndarray]:

    """ 
    Measures how accuracy a model classifies unseen emails, using stratified k-fold cross validation. 

    Args:
        model_to_test: a class or function that makes a new untrained model.
        X: features, shape (n, 54)
        y: labels, shape (n, )
        seed: saves the starting point of the random shuffle for reproducibility 
        k: the number of folds
        verbose: when True will print the accuracy for each fold and summary statistics for all folds. 
    
    Returns:
        mean: average accuracy for the k folds
        std: standard deviation of the fold accuracy results
        accuracy_per_fold: the k individual fold accuracies, shape (k,)
    """

    rng = np.random.default_rng(seed)
    folds = strat_k_fold(y, k, rng)
    accuracy_per_fold = []

    for i, test_rows in enumerate(folds):
        # train the model on all rows that are not part of the test_rows
        train_rows = np.setdiff1d(np.arange(len(y)), test_rows)

        # Create a copy of the model for each fold selected to be the test set. 
        # This model copy is only trained on the other k-1 folds. 
        model = model_to_test()
        model.train(X[train_rows], y[train_rows])

        predictions =  model.predict(X[test_rows])
        # proportion of correct predictions for this fold's test emails
        fold_accuracy = np.mean(predictions == y[test_rows]) 
        accuracy_per_fold.append(fold_accuracy)

        if verbose:
            print(f"fold {i+1}/{k}: fold accuracy = {fold_accuracy:.3f} ")
            print(f"train {len(train_rows)}, test= {len(test_rows)}")

    accuracy_per_fold = np.array(accuracy_per_fold)
    if verbose:
        print(f"mean = {accuracy_per_fold.mean():.3f}   std = {accuracy_per_fold.std():.3f}")

    return float(accuracy_per_fold.mean()), float(accuracy_per_fold.std()), accuracy_per_fold

class MostCommonClass:
    """
    Model that will always predict the most common label based on training data proportions. 

    This model will ignore the keywords and always only pick the most common label in your training data.
    The cross-validation accuracy for the model should be approximately equal to the baseline accuracy from inspect_data().count

    Every real model should aim to beat this score. 
    """
    def train(self, X: np.ndarray, y:np.ndarray) -> None:
        """
        Contains the majority label from the training data.

        If 50% or more of the training emails are spam then the majority label is spam,
        else the majority would be ham. 
        """
        self.label = int(y.mean() >= 0.5)

    def predict(self, X: np.ndarray) -> np.ndarray:
        """
        Returns a list of the majority label learned from train() for every email with shape(n, ). 
        
        Predict does not see the labels (y), only the features (X).
        """
        # make a list of length of the number of emails and filled with the majority label
        return np.full(X.shape[0], self.label)



In [8]:
RUN_TESTING = True

if RUN_TESTING:
    # stacks training and test such that data will have shape (1500, 55)
    data = np.vstack([training_spam, testing_spam])
    X, y = split_x_and_y(data)
    p_ham, p_spam = inspect_data(X, y)
    print()
    cross_validate(MostCommonClass, X, y, k=10)

1500 emails, 54 features
spam: 586 (39.1%) ham: 914 (60.9%)
a model that always predicts the more common class has baseline accuracy of: 0.609
strongest spam indicating keywords
 keyword   P(x|ham)  P(x|spam)  spam_diff_ham
       51      0.286      0.857          +0.57
       52      0.106      0.619          +0.51
       15      0.097      0.568          +0.47
       20      0.337      0.804          +0.47
        4      0.233      0.623          +0.39
        6      0.015      0.401          +0.39
       23      0.018      0.398          +0.38
        2      0.278      0.599          +0.32
       16      0.106      0.422          +0.32
       18      0.571      0.884          +0.31

strongest ham indicating keywords
 keyword   P(x|ham)  P(x|spam)  spam_diff_ham
       24      0.389      0.029          -0.36
       26      0.271      0.003          -0.27
       25      0.281      0.017          -0.26
       36      0.299      0.055          -0.24
       45      0.179      0.031      

In [ ]:
# ---Stage 2: A Bernoulli Naïve Bayes Classifier---

class NaiveBayesClassifier:
    """
    A Bernoulli Naïve Bayes spam classifier for keywords in emails.

    Model formula explained: P(class | x_1, x_2, ..., x_54) ∝ P(class) * ∏_j P(x_j | class) 
    where class is the cause variable and the features (x_1 to x_54) are the effect
    variables (Russell and Norvig, 2022, §12.6, Eq. 12.21). The naïve aspect comes from the 
    simplifying assumption that the features are independent of each other, given the class. 
    This allows the joint probability of all features to be expressed as a product of the
    individual feature probabilities, which can each be estimated from the 
    training data (Russell and Norvig, 2022, §12.6, Eq. 12.20). 

    Each feature tracks if keyword j was found in the email (x_j = 1) or not (x_j = 0), so 
    it can be modelled as a Bernoulli variable with P(x_j = 1 | class) = p and 
    P(x_j = 0 | class) = 1 - p such that P(x_j | class) = p^x_j * (1 - p)^(1 - x_j). 
    In this model, since 1 - p is included, the nonoccurrence of a keyword also affects 
    the classification (Manning, Raghavan and Schütze, 2008, §13.3, pp. 243-244;
    McCallum and Nigam, 1998). 

    The train() and predict() methods are based on Manning et al. (2008, Figure 13.3, p. 244)
    but use matrix multiplication instead of loops over the classes and keywords. 

    Attributes:
        log_prior: log P(class) where class 0 = ham and class 1 = spam, shape(2, )
        p_found: Laplace-smoothed P(keyword found | class), shape (2, 54)
        log_found: log(p_found), shape (2, 54)
        log_not_found: log(1-p_found), shape (2, 54)
    """
    def __init__(self, alpha: float = 1.0):
        """
        Args:
            alpha: Laplace smoothing count added to both the found and not found counts 
                for each keyword, for each class. 
                alpha = 1 is add-one smoothing (Russell and Norvig, 2022, §24.1.4, p. 878).
                alpha = 0 has no smoothing, so a keyword with a probability of 0 would 
                wipe out all of our other evidence. This is the case for 
                keyword number 40 which is never found in spam so any 
                email with this keyword could never be classified as spam 
                (Russell and Norvig, 2022, §12.6.1, p. 421, footnote 8).
        """
        self.alpha = alpha

    def train(self, X: np.ndarray, y: np.ndarray) -> None:
        """
       Uses the training data frequencies to estimate the class priors P(class) and the 
       conditional probabilities P(keyword found | class) for each keyword 
       (Manning et al., 2008, Figure 13.3, p. 244). 

       Args:
        X: features with values 0 or 1, shape (n, 54)
        y: labels, 0 = ham and 1 = spam, shape (n, )
        """
        n_emails, n_features = X.shape
        n_spam_emails = int(y.sum())
        n_ham_emails = n_emails - n_spam_emails
        n_each_class = np.array([n_ham_emails, n_spam_emails])

        # Prior is the proportion of the training emails that are spam vs ham,
        # Stored as a log so it can be added to the keyword probabilities (also logs)
        # in log_posterior. No smoothing needed here since both classes have lots of emails. 
        self.log_prior = np.log(n_each_class) - np.log(n_emails)

        # Build a (2, 54) table where row 0 is the number of ham emails containing 
        # each keyword, and Row 1 is the same for spam. 
        count_found = np.vstack([X[y == 0].sum(axis=0), X[y == 1].sum(axis=0)])

        # Laplace smoothing where we add alpha pseudo-counts to the numerator (found count)
        # and alpha for each of the outcomes (found and not found) to the 
        # denominator (class total) (Manning et al., 2008, Figure 13.3, p. 244). 
        # We need to do this to avoid P(found | spam) = 0 and log(0) for keyword 40. 
        self.p_found = (count_found + self.alpha) / (n_each_class[:, None] + 2* self.alpha)
        self.log_found = np.log(self.p_found)
        self.log_not_found = np.log(1 - self.p_found)

    def log_posterior(self, X: np.ndarray) -> np.ndarray:
        """
        Taking the log of the equation given in the class docstring:
        log P(class) + Σ_j + log P(x_j | class) as mentioned in 
        Manning et al. (2008, Figure 13.3, p. 244).

        We take logs here to avoid floating point underflow which can result 
        from multiplying many conditional probabilities. Since log is monotonic, 
        the larger score is still the larger probability (Manning et al., 2008, §13.2, pp. 238-239)

        Args:
            X: features, shape (n, 54)
        
        Returns:
            unnormalized log posterior with column 0 for ham and column 1 
            for spam and with shape (n, 2). It is unnormalized since P(x), the denominator, 
            is the same for both classes (Manning et al., 2008, Eq. 13.9-13.10, p. 245).   
        """
        # Using matrix multiplication rather than looping.
        return X @ self.log_found.T + (1 - X) @ self.log_not_found.T + self.log_prior

    def predict_spam_prob(self, X: np.ndarray) -> np.ndarray:
        """
        Calculates P(spam | keywords) from the log posterior. This is done by 
        reversing the logs and normalizing the ham and spam values so they sum to 1
        (Manning et al., 2008, §13.4, p. 249).

        Args:
            X: features, shape (n, 54)

        Returns:
            P(spam | keywords) per email, shape (n, ).
        """
        log_post = self.log_posterior(X)

        # NEED NOTES AND CITATION HERE
        # Goodfellow?
        log_post = log_post - log_post.max(axis=1, keepdims=True)
        probabilities = np.exp(log_post)
        return(probabilities/probabilities.sum(axis=1, keepdims=True))[:, 1]




NameError: name 'np' is not defined

In [ ]:
# This skeleton code simply classifies every input as ham
#
# Here you can see there is a parameter k that is unused, the
# point is to show you how you could set up your own. You might
# also pass in extra data via a train method (also does nothing
# here). Modify this code as much as you like so long as the 
# accuracy test in the cell below runs.

class SpamClassifier:
    def __init__(self, k):
        self.k = k
        
    def train(self):
        pass
        
    def predict(self, data):
        return np.zeros(data.shape[0])
    

def create_classifier():
    classifier = SpamClassifier(k=1)
    classifier.train()
    return classifier

classifier = create_classifier()

### Accuracy Estimate
In the cell below there is a function called `my_accuracy_estimate()` which returns `0.5`. Before you submit the assignment, write your best guess for the accuracy of your classifier into this function, as a percentage between `0` and `1`. So if you think you will get 80% of inputs correct, return the value `0.8`. This will form a small part of the marking criteria for the assignment, to encourage you to test your own code.

In [ ]:
def my_accuracy_estimate():
    return 0.5

Write all of the code for your classifier above this cell.

### Testing Details
Your classifier will be tested against some hidden data from the same source as the original. The accuracy (percentage of classifications correct) will be calculated, then benchmarked against common methods. At the very high end of the grading scale, your accuracy will also be compared to the best submissions from other students (in your own cohort and others!). Your estimate from the cell above will also factor in, and you will be rewarded for being close to your actual accuracy (overestimates and underestimates will be treated the same).

#### Test Cell
The following code will run your classifier against the provided test data. To enable it, set the constant `SKIP_TESTS` to `False`.

The original skeleton code above classifies every row as ham, but once you have written your own classifier you can run this cell again to test it. So long as your code sets up a variable called `classifier` with a method called `predict`, the test code will be able to run. 

Of course you may wish to test your classifier in additional ways, but you *must* ensure this version still runs before submitting.

**IMPORTANT**: you must set `SKIP_TESTS` back to `True` before submitting this file!

In [ ]:
SKIP_TESTS = True

if not SKIP_TESTS:
    testing_spam = np.loadtxt(open("data/testing_spam.csv"), delimiter=",").astype(int)
    test_data = testing_spam[:, 1:]
    test_labels = testing_spam[:, 0]

    predictions = classifier.predict(test_data)
    accuracy = np.count_nonzero(predictions == test_labels)/test_labels.shape[0]
    print(f"Accuracy on test data is: {accuracy}")

In [ ]:
import sys
import pathlib

fail = False;

success = '\033[1;32m[✓]\033[0m'
issue = '\033[1;33m[!]'
error = '\033[1;31m\t✗'

#######
##
## Skip Tests check.
##
## Test to ensure the SKIP_TESTS variable is set to True to prevent it slowing down the automarker.
##
#######

if not SKIP_TESTS:
    fail = True;
    print("{} \'SKIP_TESTS\' is incorrectly set to False.\033[0m".format(issue))
    print("{} You must set the SKIP_TESTS constant to True in the cell above.\033[0m".format(error))
else:
    print('{} \'SKIP_TESTS\' is set to true.\033[0m'.format(success))

#######
##
## File Name check.
##
## Test to ensure file has the correct name. This is important for the marking system to correctly process the submission.
##
#######
    
p3 = pathlib.Path('./spamclassifier.ipynb')
if not p3.is_file():
    fail = True
    print("{} The notebook name is incorrect.\033[0m".format(issue))
    print("{} This notebook file must be named spamclassifier.ipynb\033[0m".format(error))
else:
    print('{} The notebook name is correct.\033[0m'.format(success))

#######
##
## Create classifier function check.
##
## Test that checks the create_classifier function exists. The function should train the classifier and return it so that it can be evaluated by the marking system.
##
#######

if "create_classifier" not in dir():
    fail = True;
    print("{} The create_classifier function has not been defined.\033[0m".format(issue))
    print("{} Your code must include a create_classifier function as described in the coursework specification.\033[0m".format(error))
    print("{} If you believe you have, \'restart & run-all\' to clear this error.\033[0m".format(error))
else:
    print('{} The create_classifier function has been defined.\033[0m'.format(success))

#######
##
## Classifier variable check.
##
## Test that checks the classifier variable exists. The marking system will use this variable to make predictions based on a set of random features you have not seen. Your score will be based on how well your classifier predicts the hidden labels.
##
#######

if 'classifier' not in vars():
    fail = True;
    print("{} The classifer variable has not been defined.\033[0m".format(issue))
    print("{} Your code must create a variable called \'classifier\' as described in the coursework specification.\033[0m".format(error))
    print("{} This variable should contain the trained classifier you have created.\033[0m".format(error))
else:
    print('{} The classifer variable has been correctly defined.\033[0m'.format(success))

#######
##
## Accuracy Estimation check.
##
## Test that checks the accuracy estimation function exists and is a reasonable value. This is a requirement of the coursework specification and is used by the marking system when generating your final grade.
##
#######

if "my_accuracy_estimate" not in dir():
    fail = True;
    print("{} The my_accuracy_estimate function has not been defined.\033[0m".format(issue))
    print("{} Your code must include a my_accuracy_estimate function as described in the coursework specification.\033[0m".format(error))
    print("{} If you believe you have, \'restart & run-all\' to clear this error.\033[0m".format(error))
else:
    if my_accuracy_estimate() == 0.5:
        print("{} my_accuracy_estimate function warning.\033[0m".format(issue))
        print("{} my_accuracy_estimate function returns a value of 0.5 - Your classifier is no better than random chance.\033[0m".format(error))
        print("{} Are you sure this is correct.\033[0m".format(error))
    else:
        print('{} The my_accuracy_estimate function has been defined correctly.\033[0m'.format(success))

#######
##
## Test set check.
##
## Test that checks your classifier actually works. The calls made here are the same made by the automarker - albeit with different data. If your work fails this test it will score 0 in the automarker.
##
#######

try:
    testing_spam = np.loadtxt(open("data/testing_spam.csv"), delimiter=",").astype(int)
    test_data = testing_spam[:, 1:]
    test_labels = testing_spam[:, 0]
    
    try:
        predictions = classifier.predict(test_data)
        accuracy = np.count_nonzero(predictions == test_labels)/test_labels.shape[0]
        print('{0} Success running test set - Accuracy was {1:.2f}%.\033[0m'.format(success, (accuracy*100)))
    except Exception as e:
        fail = True
        print("{} Error running test set.\033[0m".format(issue))
        print("{} Your code produced the following error. This error will result in a zero from the automarker, please fix.\033[0m".format(error))
#         print("{} {}\033[0m".format(error, e))
        print(e)
except:
    sys.stderr.write("Unable to run one test as the file \'data/testing_spam.csv\' could not be found.")

#######
##
## Final Summary
##
## Prints the final results of the submission tests.
##
#######

if fail:
    sys.stderr.write("Your submission is not ready! Please read and follow the instructions above.")
else:
    print("\033[1m\n\n")
    print("╔═══════════════════════════════════════════════════════════════╗")
    print("║                        Congratulations!                       ║")
    print("║                                                               ║")
    print("║            Your work meets all the required criteria          ║")
    print("║                   and is ready for submission.                ║")
    print("╚═══════════════════════════════════════════════════════════════╝")
    print("\033[0m")
    

In [ ]:
# This is a test cell. Please do not modify or delete.